# 02 · Preparación de datos (CRISP-DM: Preparación de los datos)

ETL: se carga el CSV completo a SQLite sin modificarlo, y con SQL se construyen las tablas filtradas para el análisis RFM.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/raw/online_retail_II.csv")

# Nombres de columnas sin espacios ni mayúsculas (más cómodos en SQL)
df = df.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
})

df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["customer_id"] = df["customer_id"].astype("Int64")  # entero que admite vacíos

conn = sqlite3.connect("../data/processed/retail.db")
df.to_sql("transacciones_raw", conn, if_exists="replace", index=False, chunksize=50_000)
print("Carga completa")

Carga completa


In [2]:
pd.read_sql("""
    SELECT COUNT(*)           AS filas,
           COUNT(customer_id) AS filas_con_cliente
    FROM transacciones_raw
""", conn)

,filas,filas_con_cliente
0,1067371,824364


In [3]:
with open("../sql/01_limpieza.sql", encoding="utf-8") as f:
    conn.executescript(f.read())

pd.read_sql("""
    SELECT 'ventas_limpias' AS tabla,
           COUNT(*) AS filas,
           COUNT(DISTINCT customer_id) AS clientes,
           ROUND(SUM(total), 2) AS monto
    FROM ventas_limpias
    UNION ALL
    SELECT 'devoluciones',
           COUNT(*),
           COUNT(DISTINCT customer_id),
           ROUND(SUM(monto_devuelto), 2)
    FROM devoluciones
""", conn)

,tabla,filas,clientes,monto
0,ventas_limpias,776579,5852,17068582.72
1,devoluciones,17586,2444,709953.43


In [4]:
pd.read_sql("""
    SELECT
        CASE WHEN customer_id IS NULL THEN 'Sin cliente' ELSE 'Con cliente' END AS grupo,
        COUNT(*) AS filas,
        ROUND(SUM(quantity * price), 2) AS ventas,
        ROUND(100.0 * SUM(quantity * price) / (
            SELECT SUM(quantity * price)
            FROM transacciones_raw
            WHERE invoice NOT LIKE 'C%' AND quantity > 0 AND price > 0
        ), 1) AS pct_ventas
    FROM transacciones_raw
    WHERE invoice NOT LIKE 'C%' AND quantity > 0 AND price > 0
    GROUP BY grupo
""", conn)

,grupo,filas,ventas,pct_ventas
0,Con cliente,805549,17743429.18,84.6
1,Sin cliente,236121,3229165.39,15.4


## Impacto de los filtros

- Las ventas sin `Customer ID` son el 22.7 % de las filas pero el 15.4 % del monto vendido (£3.23 M): compras de menor valor que no pueden asignarse a un cliente. La segmentación describe solo a los clientes identificados.
- Al quitar duplicados y cargos que no son productos, el monto con cliente pasa de £17.74 M a £17.07 M (−3.8 %).
- `ventas_limpias`: 776,579 filas, 5,852 clientes y £17.07 M. Es la base del análisis RFM.
- `devoluciones`: 17,586 filas, 2,444 clientes (≈42 %) y £0.71 M (≈4.2 % de las ventas). Se usará para la tasa de devolución por cliente.
- La tabla `transacciones_raw` conserva los datos originales sin modificar.

In [8]:
with open("../sql/02_rfm.sql", encoding="utf-8") as f:
    conn.executescript(f.read())

rfm = pd.read_sql("SELECT * FROM rfm", conn)
rfm.to_csv("../data/processed/rfm.csv", index=False)

print(rfm.shape)
rfm.describe()

rfm.drop(columns="customer_id").describe().round({
    "recency": 1,
    "frequency": 1,
    "monetary": 2,
    "monto_devuelto": 2,
    "tasa_devolucion": 4,
})

(5852, 9)


,recency,frequency,monetary,monto_devuelto,tasa_devolucion
count,5852.0,5852.0,5852.00,5852.00,5852.0000
mean,199.7,6.3,2916.71,121.08,0.0241
std,208.5,12.7,14306.87,2502.98,0.0965
min,0.0,1.0,2.95,0.00,0.0000
25%,25.0,1.0,339.58,0.00,0.0000
50%,95.0,3.0,856.02,0.00,0.0000
75%,379.0,7.0,2241.03,27.40,0.0147
max,738.0,373.0,580987.04,168469.60,2.8584


## Tabla RFM

- 5,852 clientes, una fila por cliente. Fecha de referencia: 10/12/2011 (día siguiente a la última venta).
- Recency: mediana 95 días (promedio 199.7).
- Frequency: mediana 3 compras, promedio 6.3, máximo 373.
- Monetary: mediana £856, promedio £2,917, máximo £580,987.
- Frequency y Monetary están muy sesgadas: el promedio duplica o triplica la mediana por unos pocos clientes de gran volumen, probablemente mayoristas. No se eliminan; en el modelado se aplicará una transformación logarítmica.
- Algunos clientes tienen una tasa de devolución mayor a 1, probablemente porque devolvieron compras anteriores al período del dataset. Esta variable se usa solo para describir los segmentos y se limitará a 1.
- Resultado exportado a `data/processed/rfm.csv`.